<a href="https://colab.research.google.com/github/jygheo/Contrastive-Decoding/blob/main/music_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from google.colab import runtime
drive.mount('/content/drive')
import torch
from IPython.display import Audio, display
import json
from transformers import MusicgenForConditionalGeneration
from multiprocessing.dummy import Pool as ThreadPool
import scipy.linalg

In [ ]:
!pip install laion-clap frechet_audio_distance hear21passt mauve-text


In [ ]:
import os
import torchaudio
import numpy as np
import torch.nn.functional as F
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from frechet_audio_distance import FrechetAudioDistance
import hear21passt.base
import mauve
from tqdm import tqdm
from multiprocessing.dummy import Pool as ThreadPool



In [ ]:
# --- CONFIGURATION ---
EVAL_DIR = "./eval_audio"
REF_DIR = "./ref_audio"
GEN_DIR = "./gen_audio"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(REF_DIR, exist_ok=True)
os.makedirs(GEN_DIR, exist_ok=True)

def decode_tokens_to_wav(records, methods, batch_size=16):
    model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(DEVICE)
    model.eval()

    for i in tqdm(range(0, len(records), batch_size), desc="Refs"):
        batch_records = records[i:i+batch_size]
        ref_tokens = [r["human_reference"] for r in batch_records]

        ref_tensor = torch.tensor(ref_tokens).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            ref_audios = model.audio_encoder.decode(ref_tensor, audio_scales=[None]*len(ref_tokens)).audio_values.cpu()

        for j, record in enumerate(batch_records):
            torchaudio.save(f"{REF_DIR}/{record['id']}_ref.wav", ref_audios[j], 32000)

    for method in methods:
        method_dir = os.path.join(GEN_DIR, method)
        os.makedirs(method_dir, exist_ok=True)

        for i in tqdm(range(0, len(records), batch_size), desc=method):
            batch_records = records[i:i+batch_size]
            gen_tokens = [r["generations"][method] for r in batch_records]

            gen_tensor = torch.tensor(gen_tokens).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                gen_audios = model.audio_encoder.decode(gen_tensor, audio_scales=[None]*len(gen_tokens)).audio_values.cpu()

            for j, record in enumerate(batch_records):
                torchaudio.save(f"{method_dir}/{record['id']}_gen.wav", gen_audios[j], 32000)

    del model
    torch.cuda.empty_cache()

def compute_passt_kld(methods, records, batch_size=32):
    passt = hear21passt.base.get_basic_model(mode="logits").to(DEVICE)
    passt.eval()

    def get_passt_probs_batch(paths):
        all_probs = []
        for i in tqdm(range(0, len(paths), batch_size), desc="PaSST Batching", leave=False):
            batch_paths = paths[i:i+batch_size]
            wavs = []
            for p in batch_paths:
                wav, sr = torchaudio.load(p)
                if sr != 32000:
                    wav = torchaudio.functional.resample(wav, sr, 32000)
                wavs.append(wav.squeeze(0) if wav.ndim == 2 else wav)

            max_len = max(w.shape[-1] for w in wavs)
            padded_wavs = [F.pad(w, (0, max_len - w.shape[-1])) for w in wavs]
            wav_tensor = torch.stack(padded_wavs).to(DEVICE)

            with torch.no_grad():
                logits = passt(wav_tensor)
                probs = F.softmax(logits, dim=-1)
            all_probs.append(probs.cpu())

        return torch.cat(all_probs, dim=0)

    ref_paths = [f"{REF_DIR}/{record['id']}_ref.wav" for record in records]
    ref_probs = get_passt_probs_batch(ref_paths)
    ref_dist = ref_probs.mean(dim=0)

    results = {}
    for method in methods:
        gen_paths = [f"{GEN_DIR}/{method}/{record['id']}_gen.wav" for record in records]
        gen_probs = get_passt_probs_batch(gen_paths)
        gen_dist = gen_probs.mean(dim=0)

        kl_div = F.kl_div(gen_dist.log(), ref_dist, reduction='sum').item()
        results[method] = kl_div

    del passt
    torch.cuda.empty_cache()
    return results

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    diff = mu1 - mu2
    covmean, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False)

    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = scipy.linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)
    return (diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean)

def extract_vggish_features_for_fad(file_paths, vggish_model):
    def _process(path):
        with torch.no_grad():
            return vggish_model.forward(path).cpu().numpy()

    with ThreadPool(8) as pool:
        all_features = list(tqdm(pool.imap(_process, file_paths), total=len(file_paths), desc="VGGish (FAD)", leave=False))

    return np.vstack(all_features)

def compute_fad_fast(methods, records):
    vggish = torch.hub.load('harritaylor/torchvggish', 'vggish', postprocess=False)
    vggish.eval()
    vggish.to(DEVICE)

    if isinstance(vggish.embeddings[-1], torch.nn.ReLU):
        vggish.embeddings = torch.nn.Sequential(*list(vggish.embeddings.children())[:-1])

    ref_paths = [f"{REF_DIR}/{record['id']}_ref.wav" for record in records]
    human_features = extract_vggish_features_for_fad(ref_paths, vggish)
    mu_ref = np.mean(human_features, axis=0)
    sigma_ref = np.cov(human_features, rowvar=False)

    fad_results = {}

    for method in methods:
        gen_paths = [f"{GEN_DIR}/{method}/{record['id']}_gen.wav" for record in records]
        gen_features = extract_vggish_features_for_fad(gen_paths, vggish)

        mu_gen = np.mean(gen_features, axis=0)
        sigma_gen = np.cov(gen_features, rowvar=False)

        fad_score = calculate_frechet_distance(mu_ref, sigma_ref, mu_gen, sigma_gen)
        fad_results[method] = fad_score

    del vggish
    torch.cuda.empty_cache()
    return fad_results

def extract_vggish_features(file_paths, vggish_model):

    all_features = []

    # tqdm will give you a real-time progress bar and frames-per-second metric
    for path in tqdm(file_paths, desc="Extracting Audio Features", leave=False):
        with torch.no_grad():
            # forward() reads the .wav, resamples, and runs the network.
            # Output is automatically a tensor of shape (Num_Seconds, 128)
            features = vggish_model.forward(path)
            all_features.append(features.cpu().numpy())

    # Stack into a single massive array: (Total_Seconds, 128)
    return np.vstack(all_features)

def compute_vggish_mauve(methods, records):
    vggish = torch.hub.load('harritaylor/torchvggish', 'vggish')
    vggish.eval()
    vggish.to(DEVICE)

    # 1. Extract all Human Reference features (P)
    ref_paths = [f"{REF_DIR}/{record['id']}_ref.wav" for record in records]

    human_features = extract_vggish_features(ref_paths, vggish)

    mauve_results = {}

    # 2. Extract Generated features (Q) and compute MAUVE
    for method in methods:
        gen_paths = [f"{GEN_DIR}/{method}/{record['id']}_gen.wav" for record in records]

        gen_features = extract_vggish_features(gen_paths, vggish)

        num_buckets = 500

        out = mauve.compute_mauve(
            p_features=human_features,
            q_features=gen_features,
            device_id=0 if DEVICE == "cuda" else -1,
            max_text_length=1024, # Required by the API signature, but ignored
            verbose=False,
            num_buckets=num_buckets
        )

        mauve_results[method] = out.mauve

    # Cleanup
    del vggish
    torch.cuda.empty_cache()
    return mauve_results


# ==========================================
# PIPELINE EXECUTION & FORMATTING
# ==========================================
def evaluate_dataset(jsonl_path):

    with open(jsonl_path, 'r') as f:
        records = [json.loads(line) for line in f]
        methods = list(records[0]["generations"].keys())

    total_samples = len(records)

    decode_tokens_to_wav(records, methods, batch_size=16)

    kld_results = compute_passt_kld(methods, records, batch_size=32)
    fad_results = compute_fad_fast(methods, records)
    mauve_results = compute_vggish_mauve(methods, records)

    results = {}
    for method in methods:
        results[method] = {
            "KLD": kld_results[method],
            "FAD": fad_results[method],
            "MAUVE":  mauve_results[method]
        }

    print(f"\n=== FINAL RESULTS for {jsonl_path} ===")
    print(f"{'Method':<15} | {'FAD':<10} | {'MAUVE':<10} | {'KLD':<10}")
    print("-" * 55)
    for method, metrics in results.items():
        print(f"{method:<15} | {metrics['FAD']:<10} | {metrics['MAUVE']:<10} | {metrics['KLD']:<10}")

In [ ]:
JSONL_PATH = "/content/drive/MyDrive/CS4782-final/music-gen/anti-prompt-on-123/musicgen-large_vs_musicgen-small.jsonl"
evaluate_dataset(JSONL_PATH)